In [1]:
import sys
sys.version

'3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]'

# what the Risk Desk does
# fintech startup that operates a retail algorithmic trading platform. 
# Founded in 2020 by two ex-investment bankers
# serve over 14,000 active traders, executing an average of 3.2 lakh trades per month.

# what problem you are solving, 
# 23 trader accounts hit margin calls
# what the notebook will eventually produce

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl as o

packages=[
    ('numpy', 'np'),
    ('pandas', 'pd'),
    ('matplotlib', 'matplotlib'),
    ('seaborn', 'sns'),
    ('scipy', 'scipy'),
    ('sklearn', 'sklearn'),
]
for pkg, alias in packages:
    try:
        mod=__import__(pkg)
        ver= getattr(mod,'__version__','unknown')
        print(f'{pkg:<12} {alias:<14} {ver:<12}')
    except ImportError:
        print(f'{pkg:<12} {alias:<14} {'-':<12}')
        

numpy        np             2.3.5       
pandas       pd             2.3.3       
matplotlib   matplotlib     3.10.6      
seaborn      sns            0.13.2      
scipy        scipy          1.16.3      
sklearn      sklearn        1.7.2       


In [3]:
import subprocess
import sys

result= subprocess.run(
    [sys.executable,'-m','pip','show','scipy'],
    capture_output= True,
    text=True
)
if result.returncode==0:
    print('scipy is already installed')
else:
    print('scipy is not installed')

scipy is already installed


#  What breaks down when you try to do margin 
# utilisation calculations across 14,000 traders in Excel? What does Python make possible 
# that Excel structurally cannot do? 

1. Repetitive manual work

You import the data, add formulas, drag them down, apply filters, and repeat the process whenever new data arrives. This increases the chance of human error.

Python advantage: automate the entire workflow with a script.

2. Dirty or invalid data

Some traders may have missing margin values, text instead of numbers, negative balances, or zero denominators. Excel formulas can return errors or misleading results if these cases are not handled consistently.

Python advantage: systematically validate records, log errors, and handle exceptional cases.

3. Complex risk rules

Suppose you need to identify traders with utilisation above 80%, group them by desk, calculate the average utilisation, and find the ten highest-risk records.

In Excel, you can do this with formulas, filters, PivotTables, and Power Query. Python makes the process easier to combine into one repeatable pipeline.

4. Repeated daily analysis

If the risk desk receives a new file every day, repeating the same steps manually takes effort and makes consistent execution harder.

Python advantage: run the same validated calculations on every new file and produce standardised reports.

5. Scaling the analysis

Today's file may contain 14,000 traders; tomorrow's workflow may combine multiple days, desks, accounts, and transaction histories.

Python libraries such as pandas support repeatable data transformations and aggregations. Larger workloads can also be moved to databases or distributed processing systems whe

import pandas as pd
import numpy as np

df = pd.read_csv("traders.csv")

# Convert margin columns to numeric values
for col in ["margin_used", "margin_available"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Validate the records
valid = (
    df["margin_used"].notna()
    & df["margin_available"].notna()
    & (df["margin_used"] >= 0)
    & (df["margin_available"] > 0)
)

# Calculate utilisation only for valid records
df["margin_utilisation"] = np.nan

df.loc[valid, "margin_utilisation"] = (
    df.loc[valid, "margin_used"]
    / df.loc[valid, "margin_available"]
    * 100
)

# Flag traders above the illustrative 80% threshold
df["risk_flag"] = (
    df["margin_utilisation"] > 80
)

# View the 10 highest utilisation values
top_10 = df.nlargest(10, "margin_utilisation")

print(top_10)
print("Invalid records:", (~valid).sum())
print("High-utilisation traders:", df["risk_flag"].sum())

# The First Morning Check — Reading Positions with Core Python 

In [4]:
#  How many traders have unrealised losses greater than ₹50,000? 
traders=pd.read_csv('trader_positions.csv')
traders[traders.unrealised_pnl<-50000].size

496

In [5]:
#  Are there any positions where margin_available has dropped below 20% 
# of margin_used — the threshold that triggers a margin call warning?


In [6]:
# And are there any Long positions where the current_price has fallen more than 5% below avg_buy_price — a signal that 
# a stop-loss may be needed? 



In [7]:
# Open trader_positions.csv using Python's built-in open() and csv module. Print the 
# header row and the first 5 data rows to confirm the structure. Count the total number of 
# rows (positions) in the file.

import csv

with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)
    fieldnames=read.fieldnames
    print(fieldnames)
    c=0
    for i,row in enumerate(read):
        c+=1
    print(c)

['trader_id', 'trader_name', 'account_type', 'segment', 'symbol', 'position_type', 'qty', 'avg_buy_price', 'current_price', 'unrealised_pnl', 'margin_used', 'margin_available', 'position_date', 'risk_category', 'stop_loss_set', 'strategy_name']
808


In [8]:
# Loop through every row and compute the margin utilisation ratio for each position: 
# margin_used divided by (margin_used + margin_available). Handle the case where 
# margin_available is missing or stored as an empty string — treat it as zero and flag that 
# row separately. 
import csv
with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)
    fieldnames=read.fieldnames
    for i in read:
        try:
            if isinstance(float(i['margin_used']),float) and isinstance(float(i['margin_available']),float):
                i['flag']=False
            
        except ValueError as e:
            
            i['flag']=True
            used=float(str(i['margin_used']).replace(',',''))
            avail=float(str(i['margin_available']).replace(',',''))
            i['margin_utilization']=str(round(used*100/(used+avail),2))
        else:
            used=float(str(i['margin_used']).replace(',',''))
            avail=float(str(i['margin_available']).replace(',',''))
            i['margin_utilization']=str(round(used*100/(used+avail),2))
        #print(i)

In [9]:
# Write a conditional check: for each Long position, calculate how far current_price has 
# moved from avg_buy_price as a percentage. Classify each move as: 'Stop-Loss Alert' 
# (below -5%), 'Watch' (between -5% and 0%), 'Profitable' (above 0%). Print a count of 
# each category.

with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)

    counts={'Stop-Loss Alert':0,'Watch':0,'Profitable':0}
    fieldnames=read.fieldnames
    for i in read:
        try:
            if i['position_type']=='Long':
                i['far']= (float(i['current_price']) - float(i['avg_buy_price']) )*100 /float(i['avg_buy_price'])
                if i['far']<-5:
                    i['move']= 'Stop-Loss Alert' 
                elif i['far']>=-5 and i['far']<=0:
                    i['move']= 'Watch'
                elif i['far']>0:
                    i['move']= 'Profitable' 

                counts[i['move']]+=1
        except ValueError as e:
            print(f"Skipping {i['trader_id']}: {e}")
            continue

    for i,v in counts.items():
        print(f'{i:<15}: {v:<12}')

Skipping TRD_00055: could not convert string to float: ''
Skipping TRD_00282: could not convert string to float: 'ERR_NULL'
Stop-Loss Alert: 57          
Watch          : 118         
Profitable     : 208         


# pandas solution

In [10]:
import numpy as np

# Select only Long positions
long_df = df[df["position_type"] == "Long"].copy()

# Calculate percentage movement
long_df["price_change_pct"] = (
    (long_df["current_price"] - long_df["avg_buy_price"])
    / long_df["avg_buy_price"]
) * 100

# Define conditions
conditions = [
    long_df["price_change_pct"] < -5,
    (long_df["price_change_pct"] >= -5) &
    (long_df["price_change_pct"] < 0),
    long_df["price_change_pct"] > 0
]

# Define categories
categories = [
    "Stop-Loss Alert",
    "Watch",
    "Profitable"
]

# Classify each position
long_df["status"] = np.select(
    conditions,
    categories,
    default="Break-even"
)

# Print count of each category
print(long_df["status"].value_counts())

NameError: name 'df' is not defined

In [ ]:
# Use a list comprehension to extract the trader_id and unrealised_pnl for every position 
# where unrealised_pnl is more negative than -50000. Print the result as a formatted table 
# (using f-strings) showing each trader ID and their loss amount. 

with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)
    fieldnames=read.fieldnames
        
    l=[(i['trader_id'],i['unrealised_pnl']) for i in read if float(i['unrealised_pnl'])<-50000]

    print(f"{'Trader ID':<15} {'Loss Amount (₹)':>18}")
    print("-" * 35)
    for i,v in l:
        print(f'{i:<10}: {v:>18}')

In [ ]:
# Build a dictionary that maps each segment (Equity, FnO, Commodity, Currency) to the 
# total unrealised P&L across all positions in that segment. Which segment is carrying the 
# most net risk right now? 

with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)

    d={'Equity':0.0, 'FnO':0.0, 'Commodity':0.0, 'Currency':0.0}
    for i in read:
        if i['segment']=='FnO':
            d['FnO']+= float(i['unrealised_pnl'])
        if i['segment']=='Equity':
            d['Equity']+= float(i['unrealised_pnl'])
        if i['segment']=='Commodity':
            d['Commodity']+= float(i['unrealised_pnl'])
        if i['segment']=='Currency':
            d['Currency']+= float(i['unrealised_pnl'])


    most_negative=min(d,key= d.get)
    print(most_negative)

In [ ]:
# Write an expression using arithmetic operators that computes the Net Exposure for any 
#position: qty multiplied by current_price. Then compute total net long exposure (sum of 
#Net Exposure for all Long positions) and total net short exposure (sum for all Short 
#positions). Print the net market exposure (long minus short). 

with open('trader_positions.csv','r',newline='') as f:
    read=csv.DictReader(f)

    
    for i in read:
        try:
            i['net_exposure']=float(i['qty'])*float(i['current_price'])
            if i['position_type']=='Long':
                net_long += i['net_exposure']
            if i['position_type']=='Short':
                net_short +=i['net_exposure']
            
        except ValueError as e:
            print(e)

    net_market_exposure=net_long-net_short
    print(f'{net_long:.2f}' )
    print(f' {net_short:.2f}')
        

# Structuring the Risk Picture — Python Data Structures for Trading Intelligence 

In [ ]:
# Load both trader_positions.csv and trader_profile.csv into two separate lists of 
# dictionaries. Then merge them: for each position, look up the trader's profile using 
 #trader_id as the key and attach the profile fields to the position record. The result should 
# be a single list of enriched position dictionaries. 

In [ ]:
import csv
with open('trader_positions.csv','r') as f:
    read=csv.DictReader(f)
    l1=list(read)

with open('trader_profile.csv','r') as g:
    r=csv.DictReader(g)
    l2=list(r)

enriched_profile=[]

mapping={i['trader_id']:i for i in l2}

for i in l1:
    prev=i        # This does not create a new dictionary. Both prev and i refer to the same dictionary.
    if i['trader_id'] in mapping:
        data= mapping[i['trader_id']]
        prev.update(data)
        enriched_profile.append(prev)

#print(enriched_profile)


In [ ]:
# your current code only appends positions when a matching profile exists. 
# If a position has no matching profile, it will be excluded from the result.

import csv

with open('trader_positions.csv', 'r') as f:
    l1 = list(csv.DictReader(f))

with open('trader_profile.csv', 'r') as f:
    l2 = list(csv.DictReader(f))

mapping = {profile['trader_id']: profile for profile in l2}

enriched_profile = []

for position in l1:
    trader_id = position['trader_id']

    if trader_id in mapping:
        profile = mapping[trader_id]

        # Avoid overwriting the position's trader_id
        profile_fields = {
            key: value
            for key, value in profile.items()
            if key != 'trader_id'
        }

        position.update(profile_fields)

    enriched_profile.append(position)

#print(enriched_profile)

In [ ]:
#  Build a nested dictionary called risk_registry where each key is a trader_id and the value 
# is a dictionary containing: all their open positions as a list, their total unrealised P&L, 
# their margin utilisation ratio, and their margin_calls_ever from the profile. This registry 
# should allow you to look up any trader's complete risk picture in one line.
import pandas as pd
from collections import defaultdict

risk_registry = defaultdict( lambda: {
    'open_positions': [],
    'total_unrealised_pnl': 0.0,
    'margin_utilisation_ratio': None,
    'margin_calls_ever': None,
    'margin_used':0.0,
    'margin_available': 0.0
})

t=pd.read_csv('trader_positions.csv')
p=pd.read_csv('trader_profile.csv')
p

In [ ]:
t

In [ ]:
from pprint import pprint   # You can convert the nested defaultdict into a regular dictionary:

with open('trader_positions.csv','r') as f, open('trader_profile.csv','r') as g:
    positions=csv.DictReader(f)
    profiles=csv.DictReader(g)

    mapping= { i['trader_id']:i for i in profiles}
    
    for i in positions:
        trader=i['trader_id']
        pnl=float(i['unrealised_pnl'])

        risk_registry[trader]['margin_used'] +=  float(i['margin_used'].replace(',',''))
        risk_registry[trader]['margin_available'] += float(i['margin_available'].replace(',',''))
        
        risk_registry[trader]['open_positions'].append(i['position_type'])
        risk_registry[trader]['total_unrealised_pnl'] += pnl

    for trader,values in risk_registry.items():
        used = values['margin_used']
        
        available= values['margin_available']
                
        total_margin= used + available
        if total_margin>0:
            risk_registry[trader]['margin_utilisation_ratio'] = round(margin_used / total_margin,2)
        else:
            risk_registry[trader]['margin_utilisation_ratio']=0
            
    for trader, details in mapping.items():
        risk_registry[trader]['margin_calls_ever']= (details['margin_calls_ever'])
    
''' pprint({
        trader: dict(details)
        for trader, details in risk_registry.items()
    })
'''

In [ ]:
#  Holding period flag: Use the datetime module to parse position_date for every row. 
# Compute the number of calendar days the position has been open as of today. Build a 
# list of tuples — (trader_id, symbol, holding_days, stop_loss_set) — for every position 
# open longer than 30 days where stop_loss_set is 'No'. This is your Compliance Breach List. 

from datetime import datetime as dt

compliance_breach_list = []
l=[]
with open('trader_positions.csv','r') as f:
    read= csv.DictReader(f)
    for i in read:
        position= dt.strptime(i['position_date'],'%d-%m-%Y').date()
        l.append(position)

    max_date=max(l)
    for i in read:
        position= dt.strptime(i['position_date'],'%d-%m-%Y').date()
        holding_days = (max_date - position).days

        print(holding_days)

        if holding_days>30 and i['stop_loss_set']=='No':
            compliance_breach_list.append((i['trader_id'],i['symbol'],holding_days,i['stop_loss_set']))

    print(compliance_breach_list)
            

        
        
        


In [ ]:
# Use a set to find all unique symbols currently held across the entire position book. Then 
# use set intersection to find symbols that appear in both Long and Short positions 
# simultaneously — these are hedged positions and need to be reviewed separately. 

In [ ]:
with open('trader_positions.csv','r') as f:
    s=set()
    read=csv.DictReader(f)
    long=set()
    short=set()
    for i in read:
        if i['position_type'].lower() == 'long':
            s.add(i['symbol'])
            long.add(i['symbol'])
        if i['position_type'].lower() == 'short':
            s.add(i['symbol'])
            short.add(i['symbol'])
        s.add(i['symbol'])
        
    print(len(s))
    print(long.intersection(short))

In [1]:
# Build an immutable summary record for today's risk snapshot as a named tuple (or a 
# plain tuple with clear indexing): (report_date, total_positions, total_long_exposure, 
# total_short_exposure, net_exposure, traders_at_margin_risk, compliance_breaches). 
# Demonstrate why this record should be a tuple rather than a dictionary. 

from collections import namedtuple
with open('trader_positions.csv','r') as f:
    read=csv.DictReader(f)
    l=list(read)                 #likhte ho, reader already end par hota hai. Solution: CSV rows ko pehle list() mein store karo.

    # total_positions
    total_positions= len(l)
    
    
    #compliance_breaches
    max_date= max ( dt.strptime(i['position_date'],'%d-%m-%Y').date() for i in l)
    compliance_breaches=0
    risk=0
    for i in l:
        date=dt.strptime(i['position_date'],'%d-%m-%Y').date()
        holding_date=(max_date-date).days
        if (holding_date>30 and i['stop_loss_set']=='No'):
            compliance_breaches+=1

        margin_risk= float(i['margin_used'].replace(',',''))/(float(i['margin_used'].replace(',',''))+float(i['margin_available'].replace(',','')))
        if margin_risk >=0.90:
            risk+=1
    

    long_exposure=0.0
    short_exposure =0.0
    for i in l:
        try:
            exposure=float(i['qty'])*float(i['current_price'])
        except ValueError as e:
            continue
        if i['position_type'].lower()=='long':
            long_exposure += exposure
        elif i['position_type'].lower()=='short':
            short_exposure += exposure

    net_exposure= long_exposure- short_exposure

    risksnapshot= namedtuple('risksnapshot',[
        "report_date", "total_positions", "total_long_exposure",
    "total_short_exposure", "net_exposure",
    "traders_at_margin_risk", "compliance_breaches",])

    snap=risksnapshot(report_date=max_date, total_positions=total_positions, total_long_exposure=long_exposure, total_short_exposure=short_exposure,
                 net_exposure=net_exposure, traders_at_margin_risk=risk, compliance_breaches=compliance_breaches)
    print(snap)

        

NameError: name 'csv' is not defined

In [ ]:
# The Head of Risk wants a quick lookup of which strategies are being used across each 
# account type. Build a dictionary mapping account_type to a set of unique 
# strategy_names used by traders in that category.

mapping_account_type=defaultdict(set)
with open('trader_positions.csv','r') as f:
    read=list(csv.DictReader(f))

    for i in read:
        mapping_account_type[i['account_type']].add(i['strategy_name'])
    pprint({k: v for k,v in mapping_account_type.items() })

    
      

# The Broken Export — File Handling & Bulletproof Error Management 

In [ ]:
#  Write a function load_positions(filepath) that uses a context manager to read 
#trader_positions.csv. It must return two outputs: a list of valid, cleaned position records, 
#and a list of error records with a reason code attached to each. The function must not 
#raise an unhandled exception under any circumstances. 

In [ ]:
def load_positions(filepath='trader_positions.csv'):
    try:
        
        with open(filepath,'r',newline='') as f:
            read=csv.DictReader(f)
            total_columns=read.fieldnames
            column_with_errors={'trader_id','current_price'}
    
            errors=[]
            if total_columns is None or not(column_with_errors.issubset(total_columns)):
                errors.append({'line':1, 'row':None , 'reason': 'missing_columns'})
    
    
            
            invalid=[]
            valid=[]
            for i,row in enumerate(read):
                if row['trader_id'] is None or row['trader_id'].strip()=='':
                    invalid.append((i,row))
                    errors.append({'line':i ,'reason':'Missing_trader_id'})
                elif row['current_price'] is None or row['current_price']=='':
                    invalid.append((i,row))
                    errors.append({'line':i ,'reason':'Missing_current_price'})
                else:
                    valid.append((i,row))
    
            
                try:
                    price=float(row['current_price'])
                except ValueError as e:
                    errors.append({'line':i ,'reason':'Invalid_current_price'})
    
            return errors
    except FileNotFoundError:
        errors.append({'line': None, 'row': None, 'reason': 'FILE_NOT_FOUND'})
    except (UnicodeDecodeError, csv.Error):
        errors.append({'line': None, 'row': None, 'reason': 'UNREADABLE_FILE'})
    except Exception:
        errors.append({'line': None, 'row': None, 'reason': 'UNEXPECTED_ERROR'})
            
            
print(load_positions())


In [ ]:
#  Inside the function, handle every known failure mode with specific except blocks: 
# missing or extra columns (raise a custom ColumnMismatchError with the row number), 
# non-numeric values in numeric fields like margin_used, qty, unrealised_pnl (log and skip 
# the row), empty trader_id (skip and flag as 'IDENTITY_MISSING'), and duplicate rows 
# based on trader_id + symbol + position_date (keep the first, flag duplicates as 
# 'DUPLICATE_ENTRY'). 

import logging
logger= logging.getLogger(__name__)

class ColumnMismatchError(Exception):
    def _init__(self,line,msg):
        super().__init__(f'line {line}: {msg}')
        self.line=line

def load_positions(filepath='trader_positions.csv'):
    seen=set()
    errors=[]
    
    try:
        with open(filepath,'r',newline='') as f:
            read=csv.DictReader(f)
            
            total_columns=len(read.fieldnames)

            
            l=list(read)
            for i in l:

                line_no = read.line_num

                # trader missing
                trader=i['trader_id']
                if trader is None or trader.strip()=='':
                    errors.append({'line':v, 'reason':'IDENTITY_MISSING'})
                    continue
                    
                # 1. column count check
                if None in i.values():
                    raise ColumnMismatchError(line_no,'missing values in rows')
                if None in i:
                        raise ColumnMismatchError(line_no, 'extra values in row')
                

                 # 3. numeric fields
                try:
                    margin_used=float(i['margin_used'])
                    qty= float(i['qty'])
                    unrealised_pnl=float (i['unrealised_pnl'])
                except ValueError as e:
                    logger.warning(f' Non-numeric values at line %s, skipping',line_no)
                    errors.append({'line':v, 'reason':'NON_NUMERIC_VALUE'})
                    continue


            
            # duplicates
                key= ( trader, i['symbol'].strip(), i['position_date'].strip())
                if key in seen:
                    errors.append({'line':line_no, 'reason':'DUPLICATE_ENTRY'})
                seen.add(key)
                    

    except FileNotFoundError as e:
        print(e)
    except (UnicodeDecodeError,csv.Error):
        print(e)
    except Exception as e:
        print(e)

    return errors
        


print(load_positions())
            


In [ ]:
traders

In [ ]:
# After loading, use assert statements as a contract: assert that total valid rows plus error 
# rows equals total raw rows minus the header; assert that no margin_used value in the 
# valid set is negative; assert that every position_type value is either 'Long' or 'Short'. If 
# any assertion fails, raise an AssertionError with a message that tells the on-call analyst 
# exactly what went wrong. 

def load_positions(filepath='trader_positions.csv'):
    seen=set()
    errors=[]
    valid=[]
    negatives=0
    position_type=set()
    
    try:
        with open(filepath,'r',newline='') as f:
            read=csv.DictReader(f)
            line_no = read.line_num
            
            total_columns=len(read.fieldnames)

            
            l=list(read)
            for i in l:


                if i['position_type'] in position_type:
                    pass
                position_type.add(i['position_type'])

                # trader missing
                trader=i['trader_id']
                if trader is None or trader.strip()=='':
                    errors.append({'line':v, 'reason':'IDENTITY_MISSING'})
                    continue
                    
                # 1. column count check
                if None in i.values():
                    raise ColumnMismatchError(line_no,'missing values in rows')
                if None in i:
                        raise ColumnMismatchError(line_no, 'extra values in row')
                

                 # 3. numeric fields
                try:
                    margin_used=float(i['margin_used'])
                    qty= float(i['qty'])
                    unrealised_pnl=float (i['unrealised_pnl'])

                    if margin_used<0:
                        negatives+=1
                    
                except ValueError as e:
                    logger.warning(f' Non-numeric values at line %s, skipping',line_no)
                    errors.append({'line':line_no, 'reason':'NON_NUMERIC_VALUE'})
                    continue


            
            # duplicates
                key= ( trader, i['symbol'].strip(), i['position_date'].strip())
                if key in seen:
                    errors.append({'line':line_no, 'reason':'DUPLICATE_ENTRY'})
                    continue
                seen.add(key)

                valid.append(i)
    
                    

    except FileNotFoundError as e:
        print(e)
    except (UnicodeDecodeError,csv.Error):
        print(e)
    except Exception as e:
        print(e)

    return errors,valid,len(l),negatives,position_type

error,valid,actual,negative,position_type=load_positions()
assert len(error) + len(valid) == actual ,\
        'valid + invalid records are not equal'
assert negative==0 , 'there are some negative margin_used'
assert all(r['position_type'] in ('Long','Short') for r in valid), \
        'position_type other than Long or Short found in valid rows'


In [ ]:
# Write all error records to tradeiq_load_errors.csv in append mode, with columns: 
# timestamp, source_file, row_number, raw_content, error_reason. This log must 
# accumulate across runs so the tech team can see if a particular row keeps failing. 

In [ ]:
from datetime import datetime as dt

def load_positions(filepath='trader_positions.csv'):
    source=filepath
    time= dt.now().isoformat(timespec='seconds')

    errors=[]
    with open(filepath,'r',newline='') as f:
        read=csv.DictReader(f)
        
        seen=set()
        column=read.fieldnames
        for i in read:
            line_no=read.line_num

            if None in i or None in i.values():
                errors.append({'timestamp': time, 'source_file': source, 'row_number': line_no, 'raw_content': str(i), 'error_reason': 'MISSING_COLUMNS'})
                continue
            
            try:
                margin_used= float(i['margin_used'])
                qty= float(i['qty'] )
                unrealised_pnl = float(i['unrealised_pnl'])
            except ValueError as e:
                errors.append({'timestamp': time, 'source_file': source, 'row_number': line_no, 'raw_content': str(i), 'error_reason': 'non-numeric'})
                continue

            trader=i['trader_id']
            if trader is None or trader.strip()=='':
                errors.append({'timestamp': time, 'source_file': source, 'row_number': line_no, 'raw_content': str(i), 'error_reason': 'IDENTITY_MISSING'})
                continue

            check = (trader + i['symbol'].strip() + i['position_date'].strip())
            if check in seen:
                errors.append({'timestamp': time, 'source_file': source, 'row_number': line_no, 'raw_content': str(i), 'error_reason':'DUPLICATE_ENTRY'})
                continue
            seen.add(check)

        if errors:
            fieldnames=errors[0].keys()
            try:
                with open('tradeiq_load_errors.csv','a',newline='') as g:
                    write=csv.DictWriter(g,fieldnames=fieldnames)
                    write.writeheader()
                    write.writerows(errors)
            except Exception as e:
                print(e)



            
load_positions()       
            

            
            
            
        

In [ ]:
#  Safe output rule: Write a function write_risk_summary(data, output_path) that 
# computes the morning summary metrics and writes them to the output file only after all 
# computations complete successfully. Use a try/finally block so that if any computation 
# fails midway, the output file is deleted (not left as a partial write) and an alert message is 
# printed to the console.

In [ ]:
import os

def write_risk_summary(data, output_path) :
    success=False
    try:
        with open(data,'r') as g:
            read=csv.DictReader(g)

            total_no_positions=len(list(read))
            long=0
            short=0
            for i in read:

                total_margin += float(i['margin_used'])
                total_unrealised_pnl += float(i['unrealised_pnl'])

                # number of Long vs Short positions
                if i['position_type'].lower() == 'long':
                    long += 1
                elif i['position_type'].lower() == 'short':
                    short += 1

        with open(output_path,'w',encoding='utf-8') as f:
            f.write(f'total_positions: {total_no_positions}\n')
            f.write(f'total_margin: {total_margin}\n')
            f.write(f'total_unrealised_pnl: {total_unrealised_pnl}\n')
            f.write(f'total long position: {long}\n')
            f.write(f'total short position: {short}\n')
        success=True
    except Exception as e:
        print(e)
    finally:

        if not success:
            if os.path.exists(output_path):
                os.remove(output_path)
            print(f'risk summary was failed {output_path} is now deleted')
        

    


write_risk_summary('trader_positions.csv','output.csv')

                
                


        
            
    

# The Risk Engine — Functions, Lambda & a Position Intelligence Class 

In [ ]:
%%writefile tradeiq_risk_engine.py

import numpy as np

#  Module — tradeiq_risk_engine.py: Build a module with these functions: 
'''compute_margin_utilisation(margin_used, margin_available) — returns the 
utilisation ratio and raises a ValueError if either input is negative; 
classify_risk(utilisation_ratio, unrealised_pnl, stop_loss_set) — returns 
'Critical', 'High', 'Medium', or 'Low' using the official TradeIQ thresholds (document the 
thresholds in the docstring); compute_portfolio_var(*pnl_values, 
confidence=0.95) — accepts any number of P&L values and a confidence level, 
returns the Value-at-Risk estimate (the loss threshold at the given confidence 
percentile). 
'''



def compute_margin_utilisation(margin_used, margin_available):
    try:
        a= float(str(margin_used).replace(',', ''))
        b= float(str(margin_available).replace(',', ''))

        if a<0 or b<0 :
            raise ValueError ('either input is negative')
        else:
            return  a/(a+b)
        
    except ValueError as e:
        print(e)

        
    

        
def classify_risk(utilisation_ratio, unrealised_pnl, stop_loss_set):
    '''  traders with  unrealised losses greater than ₹50,000? 
         any positions where margin_available  < 20% of margin_used — the threshold that triggers a margin call warning?
         Long positions where the current_price has fallen more than 5% below avg_buy_price — a signal that a stop-loss may be needed?
         classify them as 'Critical', 'High', 'Medium', or 'Low' 
    '''
    """
        TradeIQ risk thresholds:
        > 0.90  -> Critical
        > 0.75  -> High
        > 0.25  -> Medium
        <= 0.25 -> Low
    """
    
    if utilisation_ratio > 0.90:
        return "Critical"
    elif utilisation_ratio > 0.75:
        return "High"
    elif utilisation_ratio > 0.25:
        return "Medium"
    else:
        return "Low"

        
def compute_portfolio_var(*pnl_values, confidence=0.95):
    # Value-at-Risk estimate (the loss threshold at the given confidence percentile)
    var_threshold= np.percentile(pnl_values,(1-confidence)*100)
    return var_threshold
        


    
            
            
            
            
        
            

        

In [ ]:
import tradeiq_risk_engine as mod

with open('trader_positions.csv','r',newline='') as f:
    read=list(csv.DictReader(f))

    pnl_values=[]
    for i in read:
        trader=i['trader_id']
        margin_used= i['margin_used']
        margin_available = i['margin_available']

        utilization_ratio = mod.compute_margin_utilisation(margin_used,margin_available)

        risk=mod.classify_risk(utilization_ratio,i["unrealised_pnl"],i['stop_loss_set'])

        pnl_values.append(float(i['unrealised_pnl'].replace(',','')))

        print(f' {trader:<6} has {utilization_ratio:<6} with {risk:<6}')
    
    
    var=mod.compute_portfolio_var(*pnl_values)

    


In [ ]:
# Lambda & higher-order functions: Use a lambda to sort the enriched position list by 
#unrealised_pnl ascending (worst losses first). Use map() to apply 
#compute_margin_utilisation across all positions. Use filter() to extract only positions 
#classified as 'Critical' or 'High'. Use a lambda with sorted() to produce a leaderboard of 
#the top 10 largest gross exposures (qty × current_price). 

In [ ]:
import tradeiq_risk_engine as mod

with open('trader_positions.csv','r') as f, open('trader_profile.csv','r') as g:
    read= csv.DictReader(f)
    new= csv.DictReader(g)
    enriched=[]

    mapping={ i['trader_id']: i for i in new}

    for i in read:
        new=i
        if i['trader_id'] in mapping:
            profile= mapping[i['trader_id']]

        new.update(profile)

        enriched.append(new)


    sorted_list=sorted(enriched,key= lambda x: float(x['unrealised_pnl']))



with open('trader_positions.csv','r') as f:
    read= csv.DictReader(f)

    for i in read:
        
        utilization= map(lambda x: mod.compute_margin_utilisation(x),i) 
                         
        positions=list(filter(lambda x: True if mod.classify_risk(x) in ['Critical','High'] else False, utilization))

    new_list=sorted(read,lambda x: float(x['qty'])*float(x['current_price']))

    


            
        

        


    
    
        

In [ ]:
# TradeIQPosition class: Define a class that accepts a position dictionary in __init__ and 
#stores all fields as attributes. Add methods: .gross_exposure() — returns qty × 
#current_price; .holding_days() — computes days since position_date using datetime; 
#.is_compliant() — returns False if holding_days > 30 and stop_loss_set is 'No'; 
#.risk_score() — calls the module's classify_risk() and returns the label; 
#.to_alert_dict() — returns a dictionary suitable for sending as a JSON alert to the 
#compliance system. 

In [ ]:
%%writefile class_file.py
from datetime import datetime as dt
import tradeiq_risk_engine as mod


class TradeIQPosition:
    def __init__(self,positional):
        for k,v in positional.items():
            setattr(self,k,v)

    def gross_exposure(self):
        return float(self.qty)* float(self.current_price)

    def holding_days(self):
        max_date= max( dt.strptime(self.position_date,'%d-%m-%Y') for i in position)

        holding_days= max_date - dt.strptime(self.position_date,'%d-%m-%Y')
        return holding_days

    def is_compliant(self):
        if holding_days>30 and self.stop_loss_set=='No':
            return False
        return True
    def risk_score(self):
        used=self.margin_used
        available=self.margin_available

        pnl=self.unrealised_pnl
        loss=self.stop_loss_set
        utilization_ratio= mod.compute_margin_utilisation(used,available)
        return mod.classify_risk(utilization_ratio,pnl,loss)

    def to_alert_dict():
        return {
            'trader_id':self.trader_id,
            'position': self.position_type,
            'risk_score':  self.risk_score(),
            'complaint': self.is_compliant(),
            'holding_days': self.holding_days(),
            'gross_exposure': self.gross_exposure()
        }

    
    

In [ ]:
# Demonstrate a closure: write a function make_threshold_checker(threshold) that returns 
#a new function. The returned function should accept a position dictionary and return True 
#if the margin utilisation ratio exceeds the given threshold. Show how this allows Sneha 
#(FnO desk, threshold 0.80) and Arjun (commodity desk, threshold 0.70) to each have 
#their own checker without duplicating any logic. 

In [ ]:
def make_threshold_checker(threshold):
    def check_dictionary(i):
        used= i['margin_used']
        available= i['margin_available']

        utilization_ratio= used/(used+available)
        if utilization_ratio> threshold:
            return True
        else:
            return False
    return check_dictionary

sneha_checker= make_threshold_checker(0.80)
arjun_checker= make_threshold_checker(0.70)

In [ ]:
#  In your main notebook, import tradeiq_risk_engine and the TradeIQPosition class. 
# Instantiate position objects for the top 10 critical positions. Print their risk score, gross 
# exposure, holding days, and compliance status in a formatted table.

In [ ]:
import class_file
import tradeiq_risk_engine as mod
import csv

with open('trader_positions.csv','r') as f:
    read= csv.DictReader(f)

    critical=[]
    for i in read:
        
        ratio=mod.compute_margin_utilisation(i['margin_used'],i['margin_available'])
        value= mod.classify_risk(ratio,i['unrealised_pnl'],i['stop_loss_set'])

        if value=='Critical':
            i.update({'risk':ratio})
            critical.append(i)

    sorted_critical= sorted(critical, key= lambda x: x['risk'], reverse=True)
    top_10= sorted_critical[:10]


object_list=[]
for i in top_10:
    object_list.append(class_file.TradeIQPosition(i))

print(f'trader_id | risk_score | gross_exposure | holding_days | compliance_system' )
for i in object_list:
    print(f'{i.trader_id:<10} | {i.risk_score()} | {i.gross_exposure()} | {i.holding_days()} | {i.is_compliant()}')

# Speed Under Pressure — NumPy for Real-Time Risk Metrics 

In [14]:
#  Build the arrays: Extract six columns from your cleaned position data into separate 
#NumPy arrays: qty, avg_buy_price, current_price, mmargin_available, and 
#unrealised_pnl. Verify shape, dtype, and null count for each. Convert any integer arrays 
#to float64 for consistent arithmetic. 
import numpy as np
with open('trader_positions.csv','r') as f:
    read=list(csv.DictReader(f))

    try:

        qty= np.array([float(i['qty']) for i in read],dtype= np.float64)
        avg_buy_price = np.array([float(i['avg_buy_price']) for i in read],dtype=np.float64)
        current_price = np.array([float(i['current_price']) for i in read],dtype=np.float64)
        margin_used = np.array([float(i['margin_used']) for i in read],dtype=np.float64)
        margin_available = np.array([float(i['margin_available']) for i in read],dtype=np.float64)
        unrealised_pnl = np.array([float(i['unrealised_pnl']) for i in read],dtype=np.float64)
    
    except ValueError as e:
        print(e)

    arrays = {
    'qty': qty,
    'avg_buy_price': avg_buy_price,
    'current_price': current_price,
    'margin_used': margin_used,
    'margin_available': margin_available,
    'unrealised_pnl': unrealised_pnl
    }

    for name, arr in arrays.items():
        print(
            f'{name:<20} '
            f'shape={arr.shape} '
            f'dtype={arr.dtype} '
            f'nulls={np.isnan(arr).sum()}'
        )

    for i in arrays.values():
        if np.issubdtype(i.dtype, np.interger):
            i=i.astype(np.float64)
        

could not convert string to float: '#VALUE!'


NameError: name 'current_price' is not defined

In [ ]:
#  Vectorised P&L and exposure: Compute three arrays — gross_exposure (qty × 
# current_price), cost_basis (qty × avg_buy_price), and realised_move_pct ((current_price - avg_buy_price) / avg_buy_price × 100) —
# using broadcasting, with no loops. Verify 
# that your unrealised_pnl array matches gross_exposure minus cost_basis within a 
# tolerance of ₹1. 

In [39]:
import numpy as np
import csv

with open('trader_positions.csv','r') as f:
    read=list(csv.DictReader(f))
    np.set_printoptions(suppress=True)
    try:
        qty=np.array([float(i['qty']) for i in read])
        price=np.array([ 0 if i['current_price'] in ('#VALUE!','','ERR_NULL') else float(i['current_price']) for i in read ])

        unrealised_pnl= np.array([i['unrealised_pnl'] for i in read])
        avg_buy_price = np.array([float(i['avg_buy_price']) for i in read])
        
        gross_exposure= qty*price

        cost_basis = qty* avg_buy_price

        realised_move_pct = ((price - avg_buy_price) / avg_buy_price *100)
        

    except ValueError as e:
        print(e)
    else:
        print()
        #print(gross_exposure)
        # print(avg_buy_price)
        # print(realised_move_pct)
        # print(unrealised_pnl,(gross_exposure - cost_basis))

    
    
    

In [52]:
#  Margin risk classification: Compute the margin utilisation ratio array. Use np.where() 
# to classify every position simultaneously: 'Critical' where utilisation > 0.90, 'High' where > 
# 0.75, 'Medium' where > 0.50, 'Low' otherwise. (Use nested np.where() calls.) Count how 
# many positions fall into each band using np.sum() with boolean masks. 

with open('trader_positions.csv','r') as f:
    read=list(csv.DictReader(f))

    used= np.array([float(i['margin_used'].replace(',','')) for i in read])
    available= np.array([float(i['margin_available'].replace(',','')) for i in read])

    utilization_ratio = used/(used+available)
    category = np.where(utilization_ratio>0.90 , 'Critical',np.where(utilization_ratio>0.75,'High',np.where(utilization_ratio>0.50,'Medium','Low')))

    print(f'Critical: {np.sum(category=='Critical')}')
    print(f'High: {np.sum(category=='High')}')
    print(f'Medium: {np.sum(category=='Medium')}')
    print(f'Low : {np.sum(category=='Critical')}')

Critical: 63
High: 113
Medium: 249
Low : 63


In [81]:
 # Volatility-adjusted Value-at-Risk: From market_data.csv, compute the 20-day daily 
# return series for each symbol. Load these into a 2D NumPy array of shape 
# (num_symbols, 20). Compute the standard deviation of returns for each symbol (np.std 
# across axis=1). For each position, compute a simplified 1-day VaR: gross_exposure × 
# volatility × 1.645 (the 95th percentile z-score). Sum across all positions for total desk 
# VaR.
with open('market_data.csv','r') as f:
    read= list(csv.DictReader(f))

    symbol=set([i['symbol'] for i in read])
    data={}
    for i in read:
        s=i['symbol']
        if s not in data:
            data[s]=[]
        if i['close'] == '':
            continue
        data[s].append(float(i['close']))

    returns={}

    for symbol,price in data.items():
        price= np.array(price)

        daily_returns= (price[1:] - price[:-1]) / price[:-1]    # for percentage_change in each days returns
        returns[symbol]= daily_returns[-20:] 

    return_array= np.array(list(returns.values()))

    #print(return_array)

    # Calculate volatility
    volatility = np.std(return_array,axis=1)
    #print(std_of_each_symbol)

with open('trader_positions.csv','r') as f:
    positions= list(csv.DictReader(f))


volatility_map = dict(
    zip(returns.keys(), volatility)
)

total_var=0

for pos in positions:
    symbol=pos['symbol']
    try:
        
        qty= float(pos['qty'])
        current_price = float(pos['current_price'])
    
        gross_exposure= qty* current_price
    except ValueError as e:
        print(e)
    else:
        volatile= volatility_map[symbol]
    
        var= gross_exposure*volatile*1.645
        total_var += var
print('total_var=',total_var)
    
    
    

could not convert string to float: '#VALUE!'
could not convert string to float: ''
could not convert string to float: 'ERR_NULL'
could not convert string to float: '#VALUE!'
could not convert string to float: '#VALUE!'
could not convert string to float: ''
could not convert string to float: ''
total_var= 2634203.810216205


In [101]:
# Concentration risk: Reshape the gross exposure array into a 2D array grouped by 
# segment (4 segments × N positions per segment). Use np.sum(axis=1) to get total 
#exposure per segment. Compute what percentage of total exposure each segment 
# represents. Flag any segment exceeding 40% of total exposure as a concentration risk 
# breach.

with open('trader_positions.csv','r') as f:
    read= list(csv.DictReader(f))
    segment= set([i['segment'] for i in read])
    data={}
    for i in read:
        seg=i['segment']
        try:
            
            qty=float(i['qty'])
            current_price= float(i['current_price'])
            gross_exposure=qty*current_price
        except ValueError as e:
            print(e)
        else: 
            if seg not in data:
                data[seg]= []
            data[seg].append(gross_exposure)

    sum_segments=[]
    for seg,value in data.items():
        sum_segments.append(np.sum(value))

    total_exposure=np.sum(sum_segments)
    print(total_exposure)
    percent= (np.divide(sum_segments,total_exposure)*100).round(2)
    print(percent)

    print('breach:',percent[percent>40])

could not convert string to float: '#VALUE!'
could not convert string to float: ''
could not convert string to float: 'ERR_NULL'
could not convert string to float: '#VALUE!'
could not convert string to float: '#VALUE!'
could not convert string to float: ''
could not convert string to float: ''
295178099.13000005
[12.69 74.65  7.51  5.16]
breach: [74.65]


In [120]:
#Speed test: Run your full NumPy pipeline on the 10,000-row dataset and time it using 
#Python's time module. Then simulate what 50,000 rows would look like by tiling your 
#array (np.tile) and re-run. Does it still complete under 30 seconds? Document the result.

import time

start=time.time()

with open('trader_positions.csv','r') as f:
    read= list(csv.DictReader(f))
    segment= set([i['segment'] for i in read])
    data={}
    for i in read:
        seg=i['segment']
        try:
            
            qty=float(i['qty'])
            current_price= float(i['current_price'])
            gross_exposure=qty*current_price
        except ValueError as e:
            pass
        else: 
            if seg not in data:
                data[seg]= []
            data[seg].append(gross_exposure)

    sum_segments=[]
    for seg,value in data.items():
        sum_segments.append(np.sum(value))

    total_exposure=np.sum(sum_segments)
    #print(total_exposure)
    percent= (np.divide(sum_segments,total_exposure)*100).round(2)
    #print(percent)

    #print('breach:',percent[percent>40])
end= time.time()

difference_10000= end-start

# for 50,000 rows

start_50000=time.time()
with open('trader_positions.csv','r') as f:
    read= list(csv.DictReader(f))
    segment= set([i['segment'] for i in read])
    data={}
    for i in read:
        seg=i['segment']
        try:
            
            qty=float(i['qty'])
            current_price= float(i['current_price'])
            gross_exposure=qty*current_price
        except ValueError as e:
            pass
        else: 
            if seg not in data:
                data[seg]= []
            data[seg].append(gross_exposure)

    sum_segments=[]
    for seg,value in data.items():
        sum_segments.append(np.sum(value))

    g=[]
    for gross in data.values():
        g.extend(gross)
    gross= np.array(g)
    #print(gross.shape)
    gross_50000= np.tile(gross, 50000 // len(gross) + 1)[:50000]
    
    #print(gross_50000.shape)

    total_exposure=np.sum(gross_50000)

end_50000= time.time()

diff_50000= end_50000-start_50000


if diff_50000<30:
    print('50,000 records processed under 30 sec')
else:
    print('it was not able to process under 30 sec')
    
    


50,000 records processed under 30 sec


# ``The 9 AM Report — AI-Assisted Coding for Production Deployment 

In [121]:
# AI-generated briefing formatter: Write a prompt for ChatGPT that asks it to generate a 
# function format_morning_briefing(risk_data) that accepts a dictionary of risk metrics and 
# writes a neatly formatted Morning Risk Briefing to morning_risk_report.txt. The briefing 
# must include: report timestamp, total positions, net market exposure, total VaR, traders 
# in Critical/High band, compliance breaches, and a VaR Spike Alert if triggered. Review 
# the generated code before running it — does it handle the case where risk_data has 
# missing keys? Does it produce the correct sign for net exposure? 

In [ ]:
from datetime import datetime


def format_morning_briefing(risk_data):

    # Get values safely — missing keys will not cause KeyError
    total_positions = risk_data.get('total_positions', 0)
    net_exposure = risk_data.get('net_market_exposure', 0)
    total_var = risk_data.get('total_var', 0)
    critical_high = risk_data.get('traders_critical_high', [])
    compliance_breaches = risk_data.get('compliance_breaches', [])
    var_spike = risk_data.get('var_spike_alert', False)

    # Keep the sign of net exposure
    if net_exposure > 0:
        exposure_text = f"+{net_exposure:,.2f}"
    else:
        exposure_text = f"{net_exposure:,.2f}"

    # VaR Spike Alert
    if var_spike:
        alert = "ALERT: VaR Spike Detected!"
    else:
        alert = "No VaR Spike Alert."

    # Create report
    report = f"""
========================================
        MORNING RISK BRIEFING
========================================

Report Timestamp       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

Total Positions        : {total_positions}
Net Market Exposure    : {exposure_text}
Total VaR              : {total_var:,.2f}

Traders in Critical/
High Band              : {len(critical_high)}

Compliance Breaches    : {len(compliance_breaches)}

VaR Spike Alert        : {alert}

========================================
"""

    # Write to file
    with open('morning_risk_report.txt', 'w') as f:
        f.write(report)

    print("Morning Risk Briefing generated successfully.")